# A2.6 · The agentic gateway

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A2.5 · Delegation that survives audit](https://spbreed.github.io/cyber-commons/lessons/A2.5.html)**.

| | |
|---|---|
| Open-source tooling | agentgateway, kmcp |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A2.5 works when the downstream system understands token exchange. Most do not.

Your estate contains services that accept exactly one thing — a bearer token, an
API key, a mutual-TLS client certificate — and have no field for "who is this
being done on behalf of". They will not gain one. Some are vendor products, some
are twenty years old, some are perfectly modern and simply do not implement RFC
8693.

This is where an **agent gateway** earns its place. The gateway is a single
mediation point that:

1. **Terminates the rich identity.** It receives the full on-behalf-of token,
   validates the chain, and applies policy while it still has the information.
2. **Translates down.** It calls the downstream with whatever that system does
   understand — often a narrow service credential the *gateway* holds, never the
   agent.
3. **Keeps the chain.** The act chain is recorded in the gateway's log, so the
   information is not lost even though the downstream never saw it.

The critical design point: after translation, the downstream cannot tell one
agent from another. So **every decision that depends on the chain has to happen
at the gateway**, before the identity is flattened. A gateway that only
authenticates and forwards is not a gateway, it is a proxy.

## 2 · Demo — the estate as it actually is

Three downstream systems with three different identity capabilities. Only one speaks on-behalf-of.

In [ ]:
import time, hashlib
from dataclasses import dataclass, field

@dataclass
class Token:
    sub: str; actor: str; scopes: set; act: dict = None
    issued: float = field(default_factory=time.time); ttl: float = 300
    @property
    def expired(self): return time.time() - self.issued > self.ttl
    def chain(self):
        out, node = [], self.act
        while node:
            out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub: c.insert(0, self.sub)
        return c

DOWNSTREAM = {
  "internal-api":  {"understands": "obo",     "note": "modern, RFC 8693 aware"},
  "github":        {"understands": "bearer",  "note": "PAT or app token, one identity"},
  "mainframe-fin": {"understands": "svc-acct","note": "fixed service account, 1998"},
}
for name, d in DOWNSTREAM.items():
    print(f"{name:15s} accepts {d['understands']:9s} — {d['note']}")

agent_token = Token(sub="dana@corp", actor="patch-agent", scopes={"repo:write", "ledger:post"},
                    act={"actor": "orchestrator", "act": None})
print(f"\nagent presents: {' → '.join(agent_token.chain())}  scopes={sorted(agent_token.scopes)}")

## 3 · Where it breaks — forward the token and information dies

The naive integration forwards whatever it has. Watch what each downstream can record.

In [ ]:
GATEWAY_CREDENTIALS = {          # credentials the GATEWAY holds, not the agent
    "github":        "ghs_gateway_scoped_to_repo_write",
    "mainframe-fin": "SVCACCT-GW-01",
}

def naive_forward(token, target):
    kind = DOWNSTREAM[target]["understands"]
    if kind == "obo":
        return {"target": target, "sees_actor": token.actor,
                "sees_principal": token.sub, "chain": token.chain(),
                "audit_ok": True}
    # bearer / svc-acct: there is nowhere to put the chain
    return {"target": target, "sees_actor": GATEWAY_CREDENTIALS.get(target, "?"),
            "sees_principal": "unknown", "chain": "LOST",
            "audit_ok": False}

for target in DOWNSTREAM:
    r = naive_forward(agent_token, target)
    print(f"{target:15s} actor={str(r['sees_actor'])[:26]:28s} "
          f"principal={r['sees_principal']:10s} chain={r['chain']}")
print("\nTwo of three downstreams cannot record who caused the action.")
print("Worse: to them, every agent looks like the same service account —")
print("so a per-agent policy CANNOT be enforced there. It has to happen earlier.")

## 4 · The control — decide at the gateway, then translate

The gateway applies every chain-dependent rule while it still has the chain, and only then swaps in the credential the downstream understands.

In [ ]:
POLICY = {
  # (downstream, required scope) -> which actors may reach it, and any conditions
  ("github",        "repo:write"):  {"actors": {"patch-agent"}, "max_chain": 3},
  ("mainframe-fin", "ledger:post"): {"actors": {"finance-agent"}, "max_chain": 2},
  ("internal-api",  "repo:read"):   {"actors": {"patch-agent", "triage-agent"},
                                     "max_chain": 4},
}

@dataclass
class Gateway:
    log: list = field(default_factory=list)

    def call(self, token, target, scope):
        # --- 1. decide, while the identity is still rich -----------------
        if token.expired:
            return self._deny(token, target, "token expired")
        if scope not in token.scopes:
            return self._deny(token, target, f"token lacks {scope}")
        rule = POLICY.get((target, scope))
        if rule is None:
            return self._deny(token, target, "no policy for this route (deny by default)")
        if token.actor not in rule["actors"]:
            return self._deny(token, target,
                              f"actor {token.actor} not permitted on this route "
                              f"(allowed: {sorted(rule['actors'])})")
        if len(token.chain()) > rule["max_chain"]:
            return self._deny(token, target,
                              f"delegation depth {len(token.chain())} > "
                              f"{rule['max_chain']}")
        # --- 2. translate down -------------------------------------------
        kind = DOWNSTREAM[target]["understands"]
        presented = (f"obo:{token.actor}@{token.sub}" if kind == "obo"
                     else GATEWAY_CREDENTIALS[target])
        # --- 3. keep the chain in OUR log, since downstream cannot ---------
        entry = {"allowed": True, "target": target, "scope": scope,
                 "presented_downstream": presented,
                 "chain": " → ".join(token.chain())}
        self.log.append(entry)
        return entry

    def _deny(self, token, target, why):
        entry = {"allowed": False, "target": target, "why": why,
                 "chain": " → ".join(token.chain())}
        self.log.append(entry)
        return entry

gw = Gateway()
finance = Token(sub="dana@corp", actor="finance-agent", scopes={"ledger:post"},
                act={"actor": "orchestrator", "act": None})
deep = Token(sub="dana@corp", actor="patch-agent", scopes={"repo:write"},
             act={"actor": "sub-3", "act": {"actor": "sub-2",
                  "act": {"actor": "orchestrator", "act": None}}})

for tok, target, scope, label in [
    (agent_token, "github",        "repo:write",  "permitted actor"),
    (agent_token, "mainframe-fin", "ledger:post", "wrong actor for this route"),
    (finance,     "mainframe-fin", "ledger:post", "correct actor"),
    (deep,        "github",        "repo:write",  "chain too deep"),
]:
    r = gw.call(tok, target, scope)
    print(f"{label:28s} {'ALLOW' if r['allowed'] else 'DENY '} {target:15s} "
          f"{r.get('presented_downstream', r.get('why'))}")

In [ ]:
# Verify: the chain survives in the gateway log even where the downstream
# could not carry it. This is the audit trail A2.3 said we could not reconstruct.
print("gateway audit log — the only place the full story exists:")
for e in gw.log:
    verdict = "ALLOW" if e["allowed"] else "DENY"
    print(f"   {verdict:5s} {e['target']:15s} {e['chain']}")

allowed = [e for e in gw.log if e["allowed"]]
assert all("→" in e["chain"] for e in allowed)
print(f"\n{len(allowed)}/{len(gw.log)} calls allowed; every one carries its chain,")
print("including the two whose downstream saw only a service account.")

## What you just proved

Only `internal-api` can record the actor and principal; GitHub and the mainframe both lose the chain and see a gateway credential. The gateway allows `patch-agent` to GitHub and `finance-agent` to the mainframe, denies `patch-agent` on the finance route, and denies the 4-deep chain. The gateway log carries the full chain for every call, including the two the downstream could not record.

## Your turn

List your downstream systems and mark which speak on-behalf-of. For every one that does not, name where the per-agent decision is made today. If the answer is "nowhere — they all use the same service account", you have found the gateway you need to build.

---

**Next → [A2.7 · Systems that don't understand agents](https://spbreed.github.io/cyber-commons/lessons/A2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*